# 3. Data Type Transformations and Encoding

Machine learning models require numerical input. This notebook covers:
- Ordinal encoding (for ordered categories)
- One-hot encoding (for nominal categories)
- Target and frequency encoding
- Numerical scaling (Standard, MinMax, Robust)
- Discretization (binning)
- Power transforms (Yeo-Johnson, log)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (OrdinalEncoder, OneHotEncoder,
                                   StandardScaler, MinMaxScaler,
                                   RobustScaler, PowerTransformer)

## 3.1 Load the Adult Census Dataset

In [ ]:
url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "adult/adult.data")
cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race",
        "sex", "capital_gain", "capital_loss", "hours_per_week",
        "native_country", "income"]
df = pd.read_csv(url, header=None, names=cols, na_values=" ?",
                 skipinitialspace=True)
print(f"Shape: {df.shape}")
df.head()

## 3.2 Ordinal Encoding

When categories have a **natural order** (e.g., education levels), use ordinal encoding to preserve the ranking information.

In [ ]:
edu_order = [["Preschool", "1st-4th", "5th-6th", "7th-8th", "9th",
              "10th", "11th", "12th", "HS-grad", "Some-college",
              "Assoc-voc", "Assoc-acdm", "Bachelors", "Masters",
              "Prof-school", "Doctorate"]]

enc = OrdinalEncoder(categories=edu_order,
                     handle_unknown='use_encoded_value',
                     unknown_value=-1)
df['education_ord'] = enc.fit_transform(df[['education']])
print(df[['education', 'education_ord']].drop_duplicates()
        .sort_values('education_ord'))

## 3.3 One-Hot Encoding

For **nominal** (unordered) categories, one-hot encoding creates a binary column for each category. Use `drop='first'` to avoid multicollinearity.

In [ ]:
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded = ohe.fit_transform(df[['sex', 'race']].fillna('Unknown'))
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print(f"One-hot columns: {encoded_df.columns.tolist()}")
encoded_df.head()

## 3.4 Target and Frequency Encoding

- **Target encoding**: replaces each category with the mean of the target variable for that category
- **Frequency encoding**: replaces with the frequency (proportion) of the category

Both reduce dimensionality compared to one-hot encoding.

In [ ]:
df['income_binary'] = (df['income'] == '>50K').astype(int)

# Target encoding
target_means = df.groupby('occupation')['income_binary'].mean().to_dict()
df['occupation_target'] = df['occupation'].map(target_means)

# Frequency encoding
freq = df['native_country'].value_counts(normalize=True)
df['country_freq'] = df['native_country'].map(freq)

print("Top occupations by target encoding:")
print(df[['occupation', 'occupation_target']].drop_duplicates()
        .sort_values('occupation_target', ascending=False).head(5))

## 3.5 Numerical Scaling

| Scaler | Formula | Best for |
|---|---|---|
| StandardScaler | $(x - \mu) / \sigma$ | Normal-ish data |
| MinMaxScaler | $(x - x_{min}) / (x_{max} - x_{min})$ | Bounded features |
| RobustScaler | $(x - \text{median}) / IQR$ | Data with outliers |

In [ ]:
num_features = ['age', 'hours_per_week', 'capital_gain']
data = df[num_features].fillna(0)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

for name, scaler in scalers.items():
    scaled = pd.DataFrame(scaler.fit_transform(data), columns=num_features)
    print(f"\n{name}:")
    print(scaled.describe().loc[['mean', 'std', 'min', 'max']].round(3))

## 3.6 Discretization (Binning)

Binning transforms continuous variables into categorical ones. Useful for capturing non-linear effects.

In [ ]:
# Domain-based bins
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 25, 35, 50, 65, 100],
                          labels=['<25', '25-34', '35-49', '50-64', '65+'])
print("Age groups:")
print(df['age_group'].value_counts().sort_index())
print(f"\nIncome >50K rate by age group:")
print(df.groupby('age_group')['income_binary'].mean().round(3))

## 3.7 Power Transforms

**Power transforms** (Yeo-Johnson, Box-Cox) make data more Gaussian-like by reducing skewness. This improves many algorithms that assume normality.

In [ ]:
pt = PowerTransformer(method='yeo-johnson')
df['cg_yeojohnson'] = pt.fit_transform(df[['capital_gain']])
df['cg_log'] = np.log1p(df['capital_gain'])

print(f"Original skew:     {df['capital_gain'].skew():.2f}")
print(f"Yeo-Johnson skew:  {df['cg_yeojohnson'].skew():.2f}")
print(f"Log1p skew:        {df['cg_log'].skew():.2f}")

## Key Takeaways

- **Ordinal encoding** preserves order; **one-hot** is for nominal variables
- **Target/frequency encoding** avoids the curse of dimensionality with high-cardinality features
- Choose the **scaler** based on data distribution and presence of outliers
- **Power transforms** can normalize heavily skewed distributions